# 1. Information about the submission

## 1.1 Name and number of the assignment

Assignment 3: DimABSA 2026 shared task report and reproducible experiments.

## 1.2 Student name

Denis Basharin

## 1.3 Codalab user ID / nickname / username

## 1.4 Additional comments

All experiments were run with fixed scripts in the repository root (`run_exp1.sh`, `run_exp2.sh`, `run_exp3.sh` and corresponding `test_exp*.sh`) so results are reproducible from command line.

# 2. Technical Report

*Use Section 2 to describe results of your experiments as you would do writing a paper about your results. DO NOT insert code in this part. Only insert plots and tables summarizing results as needed. Use formulas if needed do described your methodology. The code is provided in Section 3.*

## 2.1 Methodology

My submission is a transformer-based regression pipeline for Valence-Arousal (VA) prediction at aspect level. I use a BERT encoder with a lightweight regression head (`BertVARegressor`) that outputs two real values per `(text, aspect)` pair. Input formatting follows a pair setup: `"[CLS] text [SEP] aspect [SEP]"`, tokenized by Hugging Face tokenizers with truncation and dynamic padding in the batch collate function.

I trained two base configurations. **Exp1** uses `bert-base-uncased` and is expected to be stronger for English. **Exp2** uses `bert-base-multilingual-cased` and is designed for multilingual robustness. Both are trained on all available training languages (`--lang all`) with MSE loss over the 2D target `(V, A)`, AdamW optimizer, learning rate `2e-5`, batch size `64`, and maximum input length `256`. The train/dev split is loaded from the provided dataset structure through `load_track_a_subtask1_eng(...)` helper.

After observing that exp1 overfits toward English and degrades on non-English test sets, I added **Exp3** as a controlled modification of exp2. Exp3 enables a new argument `--scale_predictions` in both training and inference scripts. When this flag is active, raw model outputs are transformed as

$\hat{y}_{scaled}=4\tanh(\hat{y}_{raw})+5$

which maps predictions from $(-\infty,+\infty)$ into approximately $[1,9]$ and keeps zero centered at 5. This aligns output range with dataset score bounds and was introduced specifically to stabilize multilingual predictions without changing default behavior of previous experiments.

For reproducibility, all experiments are launched from root shell scripts: `run_exp1.sh`, `run_exp2.sh`, `run_exp3.sh` for training and `test_exp1.sh`, `test_exp2.sh`, `test_exp3.sh` for inference and metric export. Predictions are written per language into the required JSONL format, while aggregate metrics are stored in `my_code/results.json`.

## 2.2 Discussion of results


The final comparison shows three clear points. First, `exp1` remains best on English (`eng`), which is expected for an English-cased backbone. Second, `exp3` is the strongest multilingual setting overall and achieves the lowest RMSE on five languages (`jpn`, `rus`, `ukr`, `zho`, plus a near-tie on `tat`). Third, `exp2` is still slightly best on `tat` by a very small margin.

|      | eng  | jpn  | rus  | tat  | ukr  | zho  |
|------|------|------|------|------|------|------|
| exp1 | **1.01** | 1.10 | 1.65 | 1.64 | 1.65 | 0.92 |
| exp2 | 1.14 | 0.94 | 1.39 | **1.43** | 1.42 | 0.69 |
| exp3 | 1.09 | **0.88** | **1.36** | 1.43 | **1.41** | **0.66** |

Interpretation: the multilingual backbone (`exp2`) already gives a large gain over `exp1` outside English, and bounded-output scaling in `exp3` improves calibration further for most languages. Therefore, `exp3` is the best default model for multilingual evaluation, while `exp1` is preferred only if optimizing strictly for English RMSE.

# 3. Code

### Solution
This section provides a minimal and reproducible command sequence for my final pipeline.

- Environment: Python 3 + PyTorch + Transformers.
- Data: task data in `task-dataset/track_a/subtask_1`.
- Training: root scripts `run_exp1.sh`, `run_exp2.sh`, `run_exp3.sh`.
- Inference/evaluation: `test_exp1.sh`, `test_exp2.sh`, `test_exp3.sh`.
- Outputs: prediction JSONL files in language folders and summary metrics in `my_code/results.json`.

The core implementation is located in `my_code/train.py`, `my_code/test.py`, `my_code/model.py`, and `my_code/dataset.py`.


## 3.1 Requirements

In [1]:
# ### Solution
# Install dependencies used by the submitted pipeline.
!pip install -q torch transformers tqdm

^C


## 3.2 Download the data

In [2]:
# If you didn't clone the repo, run this code
# !git clone https://github.com/denisrtyhb/DimABSA2026.git
# %cd DimABSA2026

In [ ]:
# ### Solution
# Verify that the assignment dataset is available in the repository.
!ls task-dataset/track_a/subtask_1

# (Optional) if the dataset is missing, clone/fetch it before training/testing.

In [ ]:
!head task-dataset/track_a/subtask_1/eng/eng_laptop_dev_task1.jsonl

## 3.3 Preprocessing

In [ ]:
# ### Solution
# Run core experiments from repository scripts.
!source run_exp1.sh
!source run_exp2.sh
!source run_exp3.sh

In [ ]:
# ### Solution
# Run core experiments from repository scripts.
!source test_exp1.sh
!source test_exp2.sh
!source test_exp3.sh

In [12]:
# ### Solution
# Run evaluation scripts and inspect final aggregated metrics.
# !bash test_exp1.sh
# !bash test_exp2.sh
# !bash test_exp3.sh

import json
from pathlib import Path
import pandas as pd
from IPython.display import display

results_path = Path("my_code/results.json")
results = json.loads(results_path.read_text(encoding="utf-8"))

languages = ["eng", "jpn", "rus", "tat", "ukr", "zho"]
exps = ["exp1", "exp2", "exp3"]

results_df = pd.DataFrame(columns=languages)

for exp, value in results.items():
    for lang, metrics in value.items():
        results_df.loc[exp, lang] = metrics["RMSE"]

results_df = results_df.astype(float)
styled = results_df.style.format("{:.2f}").highlight_min(axis=0, props="font-weight: bold;")
display(styled)

,eng,jpn,rus,tat,ukr,zho
exp1,1.01,1.10,1.65,1.64,1.65,0.92
exp2,1.14,0.94,1.39,1.43,1.42,0.69
exp3,1.09,0.88,1.36,1.43,1.41,0.66
